In [1]:
import os, glob, json, warnings
from tqdm import tqdm
import numpy as np
import networkx as nx
import pandas as pd

from src.cell_graph_functions import *

In [2]:
# ==============================
# Graph I/O (from preprocessed NPZ)
# ==============================
def load_cell_graph_from_npz(data: np.lib.npyio.NpzFile) -> nx.Graph:
    """
    Reconstruct a NetworkX graph from edges and node count stored in a preprocessed NPZ file.
    Expects:
      - 'graph_edges'   : (E, 2) int array of undirected cell adjacencies (labels indexed 0..N-1 or any set)
      - 'graph_n_nodes' : scalar int (number of cells)
    """
    if "graph_edges" not in data.files or "graph_n_nodes" not in data.files:
        raise KeyError("NPZ must contain 'graph_edges' and 'graph_n_nodes'.")

    edges = np.asarray(data["graph_edges"]).reshape(-1, 2)
    n_nodes = int(data["graph_n_nodes"])

    G = nx.Graph()
    G.add_nodes_from(range(n_nodes))
    if edges.size:
        G.add_edges_from(edges.tolist())
    return G

# ==============================
# Curvature proxy from single HKS time
# ==============================
def hks_to_curv_proxy_single_t(hks_col, t, d=2):
    """
    Convert a single HKS column (N,) at diffusion time t to a curvature-like target:
      Y_t = (6/t) * [ (4πt)^{d/2} * HKS_t - 1 ]  ~  Gaussian curvature
    Returns float32 (N,).
    """
    h = np.asarray(hks_col, dtype=np.float64).reshape(-1)
    t = float(t)
    Y = (6.0 / t) * ((4.0 * np.pi * t) ** (d / 2.0) * h - 1.0)
    return Y.astype(np.float32)

# ==============================
# Marker filtering
# ==============================
def filter_markers(marker_names, markers_array, exclude_names):
    """
    Drop markers listed in `exclude_names` from both names and (N,M) array.
    Returns (filtered_names, filtered_markers).
    """
    if not exclude_names:
        return list(marker_names), np.asarray(markers_array, dtype=np.float32)
    keep_idx = [i for i, n in enumerate(marker_names) if n not in exclude_names]
    if not keep_idx:
        raise ValueError("After excluding markers, no markers remain. Adjust `exclude_markers`.")
    names = [marker_names[i] for i in keep_idx]
    X = np.asarray(markers_array, dtype=np.float32)[:, keep_idx]
    return names, X

# ==============================
# Detect and remove defects
# ==============================
def robust_z_scores(x):
    med = np.median(x)
    mad = np.median(np.abs(x - med)) * 1.4826  # 'normal' scale
    return (x - med) / mad if mad != 0 else np.zeros_like(x)
    

# ==============================
# Export (Project B format)
# ==============================
def _edges_from_graph(organoid_graph: nx.Graph, N_expected: int) -> np.ndarray:
    """
    Convert NetworkX graph to an undirected edge array (E,2) with u < v (int64).
    Validates node set size == N_expected.
    """
    nodes = sorted(organoid_graph.nodes())
    if len(nodes) != N_expected:
        # If nodes are labels (0..N-1) but not consecutive, reindex
        id_map = {old: new for new, old in enumerate(nodes)}
        N = len(nodes)
        warnings.warn(f"Graph node count {len(nodes)} != N={N_expected}; reindexing nodes 0..{N-1}.")
    else:
        id_map = None
        N = N_expected

    edges_uplow = set()
    for u, v in organoid_graph.edges():
        uu = id_map[u] if id_map is not None else u
        vv = id_map[v] if id_map is not None else v
        if uu == vv:
            continue
        a, b = (uu, vv) if uu < vv else (vv, uu)
        edges_uplow.add((int(a), int(b)))

    if not edges_uplow:
        return np.zeros((0, 2), dtype=np.int64)
    arr = np.array(sorted(edges_uplow), dtype=np.int64)
    # sanity
    if (arr[:, 0] < 0).any() or (arr[:, 1] >= N).any():
        raise ValueError("Edge indices out of range after reindex.")
    return arr

def export_organoid_npz(
    out_dir: str,
    organoid_id: str,
    y: np.ndarray,                   # (N,) float32
    x_bin: np.ndarray,               # (N, M) float32, binary 0/1
    organoid_graph: nx.Graph,
    marker_names=None,               # optional list[str] length M
    aux_meta: dict | None = None,    # OPTIONAL: {'total_area':..., 'total_volume':..., 'complexity':...}
):
    """
    Save one organoid in the streamlined Project (B) format:
      - x         : (N, M) float32 (binary)
      - y         : (N,)   float32
      - edges     : (E, 2) int64 (u < v)
      - N, M      : int64 scalars
    Sidecars:
      - <file>_markers.json : marker_names (if provided)
      - <file>_tvalues.json : [t_value]    (if provided)
      - <file>_aux.json     : { 'organoid_id', 'total_area', 'total_volume', 'complexity', 't_value' }
    """
    os.makedirs(out_dir, exist_ok=True)

    # Validate & coerce x,y
    X = np.asarray(x_bin, dtype=np.float32)
    if X.ndim != 2:
        raise ValueError(f"x must be 2-D (N,M); got {X.shape}")
    # Ensure {0,1}
    if not np.isin(X, [0.0, 1.0]).all():
        X = (X > 0.5).astype(np.float32)

    y = np.asarray(y, dtype=np.float32).reshape(-1)
    if y.ndim != 1:
        raise ValueError(f"y must be 1-D (N,); got {y.shape}")

    N, M = X.shape
    if y.shape[0] != N:
        raise ValueError(f"y length {y.shape[0]} != N={N}")
    if N <= 1:
        raise ValueError("N must be > 1 (need at least 2 cells).")
    if not np.isfinite(y).all():
        raise ValueError("y contains NaN/Inf.")

    # Build edges
    edge_array = _edges_from_graph(organoid_graph, N_expected=N)

    # Save NPZ (numeric only)
    file_stem = f"organoid_{organoid_id}"
    out_path = os.path.join(out_dir, f"{file_stem}.npz")
    np.savez_compressed(
        out_path,
        x=X,
        y=y,
        edges=edge_array,
        N=np.int64(N),
        M=np.int64(M),
    )

    # Sidecars
    if marker_names is not None:
        with open(os.path.join(out_dir, f"{file_stem}_markers.json"), "w") as f:
            json.dump(list(marker_names), f, indent=2)
    if aux_meta:
        save_aux_metadata(os.path.join(out_dir, file_stem), organoid_id, aux_meta)


    return out_path


def save_aux_metadata(sidecar_path_no_ext: str, organoid_id: str, aux: dict) -> None:
    """
    Save auxiliary metadata as JSON next to the NPZ. Uses a distinct filename to avoid
    interfering with the GNN loader. Only writes if `aux` is non-empty.

    Example output file: <out_dir>/organoid_<id>_aux.json
    """
    if not aux:
        return
    payload = {
        "organoid_id": str(organoid_id),
        **aux
    }
    json_path = f"{sidecar_path_no_ext}_aux.json"
    with open(json_path, "w") as f:
        json.dump(payload, f, indent=2)

In [3]:
BLACKLIST = { # Organoids with interior nodes
    "day3p5_B03_10",
    "day4_A02_44",
    "day4_A04_24",
    "day4_A04_90",
    "day4_A05_21",
    "day4_A05_53",
    "day4_B01_44",
    "day4_B01_70",
    "day4p5_A06_62",
    "day4p5_B06_128",
    "day4p5_B06_48",
    "day4p5-more_C01_5",
    "day4p5-more_C01_78",
    "day4p5-more_C02_27",
    "day4p5-more_C02_26",
    "day4p5-more_C02_35",
    "day4p5-more_C02_85",
    "day4p5-more_C02_88",
    "day4p5-more_C02_94",
    "day4p5-more_C02_127",
    "day4p5-more_C03_11",
    "day4p5-more_C03_17",
    "day4p5-more_C03_19",
    "day4p5-more_C03_45",
    "day4p5-more_C03_90",
    "day4p5-more_C03_96",
    "day4p5-more_C03_112",
    "day4p5-more_C03_137",
    "day4p5-more_C03_142",
    "day4p5-more_C04_14",
    "day4p5-more_C04_16",
    "day4p5-more_C04_22",
    "day4p5-more_C04_109",
    "day4p5-more_C05_127",
    "day4p5-more_C05_131",
    "day4p5-more_C05_142",
    "day4p5-more_C05_143",
    # the next are from Irene
    'day4p5-more_C01_138',
    'day4p5-more_C01_37', 
    'day4p5-more_C01_5',
    'day4p5-more_C01_53', 
    'day4p5-more_C01_78', 
    'day4p5-more_C01_93',
    'day4p5-more_C01_94', 
    'day4p5-more_C02_103', 
    'day4p5-more_C02_105',
    'day4p5-more_C02_127',
    'day4p5-more_C02_27',
    'day4p5-more_C02_35',
    'day4p5-more_C02_61',
    'day4p5-more_C02_77',
    'day4p5-more_C02_78',
    'day4p5-more_C02_85',
    'day4p5-more_C02_88',
    'day4p5-more_C02_94',
    'day4p5-more_C03_11',
    'day4p5-more_C03_110',
    'day4p5-more_C03_112',
    'day4p5-more_C03_135',
    'day4p5-more_C03_14',
    'day4p5-more_C03_149',
    'day4p5-more_C03_150',
    'day4p5-more_C03_152',
    'day4p5-more_C03_161',
    'day4p5-more_C03_166',
    'day4p5-more_C03_17',
    'day4p5-more_C03_171',
    'day4p5-more_C03_19',
    'day4p5-more_C03_30',
    'day4p5-more_C03_45',
    'day4p5-more_C03_60',
    'day4p5-more_C03_62',
    'day4p5-more_C03_64',
    'day4p5-more_C03_72',
    'day4p5-more_C03_86',
    'day4p5-more_C03_90',
    'day4p5-more_C03_96',
    'day4p5-more_C03_97',
    'day4p5-more_C04_109',
    'day4p5-more_C04_125',
    'day4p5-more_C04_132',
    'day4p5-more_C04_14',
    'day4p5-more_C04_16',
    'day4p5-more_C04_19',
    'day4p5-more_C04_22',
    'day4p5-more_C04_27',
    'day4p5-more_C04_30',
    'day4p5-more_C04_53',    
    'day4p5-more_C04_63', 
    'day4p5-more_C04_70', 
    'day4p5-more_C04_74',
    'day4p5-more_C04_75', 
    'day4p5-more_C04_76', 
    'day4p5-more_C04_80',
    'day4p5-more_C04_96', 
    'day4p5-more_C04_99', 
    'day4p5-more_C05_109',
    'day4p5-more_C05_127', 
    'day4p5-more_C05_131', 
    'day4p5-more_C05_142',
    'day4p5-more_C05_143', 
    'day4p5-more_C05_152', 
    'day4p5-more_C05_21',
    'day4p5-more_C05_4', 
    'day4p5-more_C05_42', 
    'day4p5-more_C05_43',
    'day4p5-more_C05_65', 
    'day4p5-more_C05_67', 
    'day4p5-more_C05_80',
    'day4p5-more_C05_82', 
    'day4p5-more_C05_94', 
    'day4p5-more_C06_100',
    'day4p5-more_C06_139', 
    'day4p5-more_C06_140', 
    'day4p5-more_C06_156',
    'day4p5-more_C06_166', 
    'day4p5-more_C06_175', 
    'day4p5-more_C06_177',
    'day4p5-more_C06_180',
}

In [4]:
# ==============================
# CONFIG (edit here)
# ==============================
save_dir = "../GraphNN/training_data"      # output dir for NPZs
include_timepoints = ["day3p5", "day4", "day4p5", "day4p5-more"]  # or None for all
exclude_markers = {}             # markers to drop
t_index = 2                                    # which HKS column to use
t_HKS = [1.0, 2.0, 4.0, 8.0, 25.0]

complexity_threshold = -1

t_index_defect = 3
defect_threshold = 10
n_defect_correct = 3


# Preprocessed base dir (Project A)
preproc_base = os.path.join("..", "NicoleData", "graphs")

tsne_df = pd.read_csv("../NicoleData/tsne_results.csv")
# Expecting columns: label_uid (e.g. "day3p5_A01_42"), ..., complexity (4th column)
complexity_dict = dict(zip(tsne_df["label_uid"], tsne_df["complexity"]))
print(f"Loaded {len(complexity_dict)} complexity scores")

# ==============================
# Discover timepoints
# ==============================
def _choose_timepoints(root, include):
    all_tps = sorted([d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))])
    return all_tps if include is None else [tp for tp in all_tps if tp in include]

timepoints = _choose_timepoints(preproc_base, include_timepoints)
print(f"Using {len(timepoints)} timepoints: {timepoints}")

# ==============================
# MAIN LOOP
# ==============================
total_exported, total_skipped = 0, 0

for tp in timepoints:
    print(f"\n=== Exporting timepoint: {tp} ===")
    # tp_dir = os.path.join(preproc_base, tp)
    # npz_files = sorted(glob.glob(os.path.join(tp_dir, "*.npz")))
    # if not npz_files:
    #     print(f"  ⚠ No .npz files found for {tp}, skipping.")
    #     continue

    # exported, skipped = 0, 0

    # for fpath in tqdm(npz_files, desc=f"{tp} (export)"):
    #     try:
    #         data = np.load(fpath, allow_pickle=True)
    #     except Exception as e:
    #         print(f"  Could not load {os.path.basename(fpath)}: {e}")
    #         skipped += 1
    #         continue

    #     # IDs
    #     well = str(data["well"])
    #     organoid_id = str(data["organoid_id"])
    #     label_uid = f"{tp}_{well}_{organoid_id}"

    tp_dir = os.path.join(preproc_base, tp)
    gpickle_files = sorted(glob.glob(os.path.join(tp_dir, "*.gpickle")))
    if not gpickle_files:
        print(f"  ⚠ No .gpickle files found for {tp}, skipping.")
        continue

    exported, skipped = 0, 0

    for fpath in tqdm(gpickle_files, desc=f"{tp}"):
        try:
            cell_graph = load_cell_graph(fpath)
        except Exception as e:
            print(f"Could not load {fpath}: {e}")
            continue
          
        label_uid = cell_graph.graph['label_uid']

        # --- Blacklist check (EARLY EXIT) ---
        if label_uid in BLACKLIST:
            skipped += 1
            continue

        # Complexity threshold
        complexity_value = complexity_dict.get(label_uid, np.nan)
        if complexity_value < complexity_threshold:
            skipped += 1
            continue

        # Arrays
        # hks_cell = np.asarray(data["fields_cell"], dtype=np.float32)   # (N, T)
        # markers_cell = np.asarray(data["markers_cell"], dtype=np.float32)  # (N, M_raw)
        # marker_names = list(data["marker_names"])

        # Graph
        # try:
        #     cell_graph = load_cell_graph_from_npz(data)
        # except Exception as e:
        #     print(f"  {label_uid}: bad graph: {e} — skipping.")
        #     skipped += 1
        #     continue


        hks_cell = np.array([cell_graph.nodes[n]["hks"] for n in cell_graph.nodes], dtype=np.float32)
        markers_cell = np.array([cell_graph.nodes[n]["markers_bin"] for n in cell_graph.nodes], dtype=np.float32)
        marker_names = ['LGR5', 'Chromogranin A', 'Cyclin D', 'Mucin 2', 'AldoB',
                'Glucagon', 'Cyclin A', 'Agr2', 'Serotonin', 'Lysozyme']

        # Basic shape checks
        if hks_cell.ndim != 2 or t_index >= hks_cell.shape[1]:
            print(f"  {label_uid}: invalid HKS shape {hks_cell.shape} for t-index {t_index}, skipping.")
            skipped += 1
            continue
        if markers_cell.ndim != 2 or markers_cell.shape[0] != hks_cell.shape[0]:
            print(f"  {label_uid}: markers shape {markers_cell.shape} incompatible with HKS {hks_cell.shape}, skipping.")
            skipped += 1
            continue

        # Filter markers
        marker_names_f, X = filter_markers(marker_names, markers_cell, exclude_markers)

        # Binarize markers
        # _, X = positive_indices_and_labels(X)
        # X = X.astype(np.float32)

        # Detect and correct defects
        zscores = robust_z_scores(hks_cell[:, t_index])
        outlier_mask = zscores > defect_threshold

        if not outlier_mask.any() or sum(outlier_mask) > n_defect_correct:
            y = hks_to_curv_proxy_single_t(hks_cell[:, t_index], t=t_HKS[t_index])  # (N,)
        else:
            # approximate curvature of defect from higher t HKS
            y = hks_to_curv_proxy_single_t(hks_cell[:, t_index], t=t_HKS[t_index])  # (N,)
            y1 = hks_to_curv_proxy_single_t(hks_cell[:, t_index_defect], t=t_HKS[t_index_defect])  # (N,)

            y[outlier_mask] = y1[outlier_mask]

        # Final guardrails
        N = X.shape[0]
        if y.shape[0] != N or N <= 1:
            print(f"  {label_uid}: invalid N or y length (N={N}, len(y)={y.shape[0]}), skipping.")
            skipped += 1
            continue
        if not np.isfinite(y).all():
            print(f"  {label_uid}: y has NaN/Inf, skipping.")
            skipped += 1
            continue

        # Build axiliary metadata
        aux_meta = {}
        # aux_meta["surface_area"] = float(np.sum(data["vertex_areas"]))
        # aux_meta["volume"] = float(data["volume"])
        aux_meta["t_value"] = float(t_HKS[t_index])
        aux_meta["timepoint"] = tp

        complexity_value = complexity_dict.get(label_uid, np.nan)
        aux_meta["complexity"] = float(complexity_value)
        
        # Export
        try:
            export_organoid_npz(
                out_dir=save_dir,
                organoid_id=label_uid,     # string ID goes into filename
                y=y,                       # (N,)
                x_bin=X,                   # (N, M_filtered)
                organoid_graph=cell_graph,
                marker_names=marker_names_f,
                aux_meta=aux_meta,
            )
            exported += 1
        except Exception as e:
            print(f"  {label_uid}: export failed: {e}")
            skipped += 1

    total_exported += exported
    total_skipped  += skipped
    print(f"  Done {tp}: exported={exported}, skipped={skipped}")

print(f"\nAll timepoints done. Total exported={total_exported}, skipped={total_skipped}")


Loaded 2668 complexity scores
Using 4 timepoints: ['day3p5', 'day4', 'day4p5', 'day4p5-more']

=== Exporting timepoint: day3p5 ===


day3p5: 100%|██████████| 312/312 [00:00<00:00, 430.26it/s]


  Done day3p5: exported=311, skipped=1

=== Exporting timepoint: day4 ===


day4: 100%|██████████| 358/358 [00:01<00:00, 264.03it/s]


  Done day4: exported=351, skipped=7

=== Exporting timepoint: day4p5 ===


day4p5: 100%|██████████| 108/108 [00:00<00:00, 191.30it/s]


  Done day4p5: exported=105, skipped=3

=== Exporting timepoint: day4p5-more ===


day4p5-more: 100%|██████████| 533/533 [00:02<00:00, 194.75it/s]

  Done day4p5-more: exported=448, skipped=85

All timepoints done. Total exported=1215, skipped=96
